# 12 – Performance tuning: broadcast vs shuffle join

Runs the same join query with two strategies (`BROADCAST`, `SHUFFLE_MERGE`) to measure the effect of the join strategy and the effect of skew separately.

Measurement: `EXPLAIN` + duration (`collect`, 3 runs, shortest time `min_s`) + Query Profile.

Tables: `workspace.perf.trans_big` and `workspace.perf.trans_skewed` (factor 100, 105.6M rows; in the skewed table, hot account 8261 has 31.4M rows).

The cell below defines the query and the measurement function and shows the `SHUFFLE_MERGE` plan.

In [0]:
import time

PERF_SCHEMA = "workspace.perf"
TABLES = ["trans_big", "trans_skewed"]

# Same query: only the join hint changes (BROADCAST or SHUFFLE_MERGE)
def join_sql(table_name, join_hint):
    sql_text = f"""
        SELECT /*+ {join_hint}(a) */
               a.district_id,
               COUNT(*) AS trans_count,
               SUM(t.amount) AS total_amount
        FROM {PERF_SCHEMA}.{table_name} t
        JOIN workspace.silver.account a
          ON t.account_id = a.account_id
        GROUP BY a.district_id
    """
    return sql_text

# 3 runs, shortest time; min removes the start-up cost of the first run
def measure(sql_text, runs=3):
    durations = []
    for run_no in range(runs):
        start = time.time()
        spark.sql(sql_text).collect()
        durations.append(time.time() - start)
    return round(min(durations), 2)

spark.sql(join_sql("trans_skewed", "SHUFFLE_MERGE")).explain()

## Broadcast vs shuffle join

The same join query is measured with 2 hints × 2 tables (4 measurements, ~1–3 min).

How to read: for the same hint, the `trans_big` ↔ `trans_skewed` difference is the **effect of skew**; for the same table, the `BROADCAST` ↔ `SHUFFLE_MERGE` difference is the **effect of the join strategy**.

Note: Photon does not support `SortMergeJoin`; the shuffle join runs row-based.

In [0]:
HINTS = ["BROADCAST", "SHUFFLE_MERGE"]

results = []
for join_hint in HINTS:
    for table_name in TABLES:
        min_s = measure(join_sql(table_name, join_hint))
        results.append((join_hint, table_name, min_s))
        print(join_hint, table_name, min_s)

df_join_results = spark.createDataFrame(results, ["join_hint", "table_name", "min_s"])
display(df_join_results)

## Results and Query Profile findings

| Hint | `trans_big` | `trans_skewed` |
|---|---|---|
| `BROADCAST` | ~1 s | ~1 s |
| `SHUFFLE_MERGE` | ~6 s | ~6 s |

Durations vary slightly between runs; the cell above shows the values of the last run.

Query Profile (`SHUFFLE_MERGE` + `trans_skewed`): the total task time is several times the wall-clock time; most of it is the join, followed by shuffle and sort. Only about a quarter of the work runs in Photon; no spill.

Comparison (`BROADCAST` + `trans_big`): almost all of the work runs in Photon, and there is no shuffle on the large side.

**Join strategy:** broadcast join is ~5× faster. In the shuffle join, 105.6M rows are shuffled and sorted by `account_id`; Photon does not support `SortMergeJoin`, so the join runs row-based.

**Skew:** for the same hint, the `trans_big` ↔ `trans_skewed` difference is small (under 10%). The profile shows no skew metric, and no AQE skew join was observed. The cost of the shuffle join comes from shuffle + sort + row-based join, not from skew.